# 02 · 多分位风险模型与因果校准

本阶段把 `direct_lgbm` 与 `shape_strength` 的每个 quantile head 视为独立 operating-point candidate。模型报告整体与 `side × ratio_bucket` coverage、pinball、final_month 评价和 crossing_rate，但不强制跨 quantile 排序。

In [ ]:
from __future__ import annotations

import json
import math
import os
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from lightgbm import LGBMRegressor


def find_repo_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / 'outputs' / '01_stage_contract.json').exists():
            return candidate
    raise FileNotFoundError('run 01_build_panel_and_labels.ipynb first')


def quantile_label(quantile: float) -> str:
    return f'q{int(round(100 * quantile)):02d}'


ROOT = find_repo_root()
OUTPUT_DIR = ROOT / 'outputs'
MODEL_DIR = ROOT / 'models'
MODEL_DIR.mkdir(parents=True, exist_ok=True)
STAGE = json.loads((OUTPUT_DIR / '01_stage_contract.json').read_text(encoding='utf-8'))
RUN_MODE = STAGE['run_mode']
QUANTILE_REGISTRY = [0.50, 0.80, 0.85, 0.90, 0.95, 0.99]
IMPACT_GUARDRAIL_QUANTILE = 0.95
SHAPE_H_MODE = 'quantile_specific_train_curve'
SHAPE_REFERENCE_X_ADV = 0.01
SHAPE_MIN_REFERENCE_SCALE_RATIO = 0.001
RANDOM_SEED = 20260807
model_panel = pd.read_parquet(ROOT / STAGE['model_panel'])
model_panel['date'] = pd.to_datetime(model_panel['date'], errors='coerce').dt.normalize()
feature_columns = list(STAGE['feature_columns'])


In [ ]:
unique_dates = sorted(model_panel['date'].dropna().unique())
if len(unique_dates) < 10:
    raise ValueError('at least 10 trading dates are required')
n_test_dates = max(1, math.ceil(len(unique_dates) * 0.20))
n_calibration_dates = max(1, math.ceil(len(unique_dates) * 0.10))
train_dates = set(unique_dates[:-(n_test_dates + n_calibration_dates)])
calibration_dates = set(unique_dates[-(n_test_dates + n_calibration_dates):-n_test_dates])
test_dates = set(unique_dates[-n_test_dates:])
if train_dates & calibration_dates or train_dates & test_dates or calibration_dates & test_dates:
    raise ValueError('time split overlap detected')
train = model_panel[model_panel['date'].isin(train_dates)].copy()
calibration = model_panel[model_panel['date'].isin(calibration_dates)].copy()
test = model_panel[model_panel['date'].isin(test_dates)].copy()

medians = train[feature_columns].apply(pd.to_numeric, errors='coerce').median().fillna(0.0).to_dict()


def make_matrix(frame: pd.DataFrame, columns: list[str] = feature_columns) -> pd.DataFrame:
    matrix = frame.reindex(columns=columns).apply(pd.to_numeric, errors='coerce')
    for column in columns:
        matrix[column] = matrix[column].fillna(float(medians.get(column, 0.0)))
    return matrix.astype('float32')


split_diagnostics = pd.DataFrame([{
    'n_dates': len(unique_dates),
    'train_min_date': min(train_dates), 'train_max_date': max(train_dates),
    'calibration_min_date': min(calibration_dates), 'calibration_max_date': max(calibration_dates),
    'test_min_date': min(test_dates), 'test_max_date': max(test_dates),
    'n_train': len(train), 'n_calibration': len(calibration), 'n_test': len(test),
    'final_month_learning_excluded': True,
}])
split_diagnostics.to_csv(OUTPUT_DIR / '02_time_split.csv', index=False)
print(split_diagnostics.to_string(index=False))


In [ ]:
def model_parameters(alpha: float) -> dict:
    return {
        'objective': 'quantile', 'alpha': float(alpha),
        'n_estimators': 45 if RUN_MODE == 'smoke' else 280,
        'learning_rate': 0.055, 'num_leaves': 31, 'min_child_samples': 80,
        'subsample': 0.90, 'colsample_bytree': 0.90,
        'reg_lambda': 1.0, 'random_state': RANDOM_SEED,
        'n_jobs': 1 if RUN_MODE == 'smoke' else -1,
        'deterministic': True, 'force_col_wise': True, 'verbosity': -1,
    }


X_train = make_matrix(train)
y_total_train = pd.to_numeric(train['total_bad_move_bps'], errors='coerce').fillna(0.0)
y_impact_train = pd.to_numeric(train['impact_me_bad_bps'], errors='coerce').fillna(0.0)
direct_models = {}
for quantile in QUANTILE_REGISTRY:
    model = LGBMRegressor(**model_parameters(quantile))
    model.fit(X_train, y_total_train)
    direct_models[quantile] = model
impact_model = LGBMRegressor(**model_parameters(IMPACT_GUARDRAIL_QUANTILE))
impact_model.fit(X_train, y_impact_train)

condition_features = [column for column in feature_columns if column != 'x_adv']
X_condition_train = make_matrix(train, condition_features)
train_x0 = train.copy()
train_x0['x_adv'] = 0.0
X_train_x0 = make_matrix(train_x0)


def build_quantile_shape_grid(frame: pd.DataFrame, increment_bad_bps: np.ndarray, quantile: float) -> pd.DataFrame:
    shape_frame = pd.DataFrame({
        'side': frame['side'].astype(str).str.lower().to_numpy(),
        'x_adv': pd.to_numeric(frame['x_adv'], errors='coerce').to_numpy(),
        'increment_bad_bps': np.maximum(np.asarray(increment_bad_bps, dtype=float), 0.0),
    }).dropna(subset=['x_adv', 'increment_bad_bps'])
    rows = []
    for side, side_frame in shape_frame.groupby('side', sort=True):
        grouped = side_frame.groupby('x_adv')['increment_bad_bps']
        curve = grouped.size().rename('n_obs').to_frame()
        curve['raw_q_bps'] = grouped.quantile(quantile)
        curve = curve.reset_index().sort_values('x_adv')
        if curve.empty or float(curve['x_adv'].iloc[0]) > 0.0:
            curve = pd.concat([pd.DataFrame([{'x_adv': 0.0, 'n_obs': 0, 'raw_q_bps': 0.0}]), curve], ignore_index=True)
        else:
            curve.loc[curve['x_adv'].eq(0.0), 'raw_q_bps'] = 0.0
        monotone = np.maximum.accumulate(np.maximum(curve['raw_q_bps'].to_numpy(dtype=float), 0.0))
        reference_scale = float(np.interp(SHAPE_REFERENCE_X_ADV, curve['x_adv'], monotone))
        max_curve_scale = float(np.max(monotone))
        reference_scale_fallback = (
            not np.isfinite(reference_scale)
            or reference_scale <= 1e-9
            or reference_scale < max_curve_scale * SHAPE_MIN_REFERENCE_SCALE_RATIO
        )
        if reference_scale_fallback:
            reference_scale = max_curve_scale
        if not np.isfinite(reference_scale) or reference_scale <= 1e-9:
            max_x = max(float(curve['x_adv'].max()), SHAPE_REFERENCE_X_ADV)
            curve['h_value'] = curve['x_adv'] / max(max_x, 1e-9)
        else:
            curve['h_value'] = monotone / reference_scale
        curve['side'] = side
        curve['quantile_level'] = quantile
        curve['reference_scale_bps'] = reference_scale
        curve['reference_scale_fallback'] = reference_scale_fallback
        rows.append(curve[['side', 'quantile_level', 'x_adv', 'n_obs', 'raw_q_bps', 'h_value', 'reference_scale_bps', 'reference_scale_fallback']])
    return pd.concat(rows, ignore_index=True)


def interpolate_quantile_shape(frame: pd.DataFrame, h_grid: pd.DataFrame) -> tuple[np.ndarray, np.ndarray]:
    values = np.zeros(len(frame), dtype=float)
    status = np.full(len(frame), 'missing_side', dtype=object)
    sides = frame['side'].astype(str).str.lower().to_numpy()
    x_values = pd.to_numeric(frame['x_adv'], errors='coerce').to_numpy(dtype=float)
    for side, side_grid in h_grid.groupby('side', sort=False):
        positions = np.flatnonzero(sides == side)
        if len(positions) == 0:
            continue
        side_grid = side_grid.sort_values('x_adv')
        support_x = side_grid['x_adv'].to_numpy(dtype=float)
        support_h = side_grid['h_value'].to_numpy(dtype=float)
        values[positions] = np.interp(x_values[positions], support_x, support_h, left=support_h[0], right=support_h[-1])
        status[positions] = np.where(x_values[positions] > support_x[-1] + 1e-12, 'right_clipped', 'in_support')
    return values, status


ANCHOR_KEY_COLUMNS = ['date', 'sym', 'side', 'quote_strategy']
ANCHOR_GRANULARITY = 'date × sym × side × quote_strategy'
STRENGTH_LABEL_GRANULARITY = 'anchor_level_curve_fit'
BASE_LABEL_SOURCE = 'direct_quantile_model_at_x0_no_observed_outcome'
ANCHOR_H_EPS = 0.05


def make_anchor_id(frame: pd.DataFrame) -> pd.Series:
    keys = frame.reindex(columns=ANCHOR_KEY_COLUMNS).copy()
    keys['date'] = pd.to_datetime(keys['date'], errors='coerce').dt.strftime('%Y-%m-%d')
    for column in ANCHOR_KEY_COLUMNS[1:]:
        keys[column] = keys[column].astype(str)
    return keys.astype(str).agg('|'.join, axis=1)


def fit_anchor_strength_label(
    frame: pd.DataFrame,
    h_values: np.ndarray,
    base_prediction_at_x0: np.ndarray,
    target_column: str,
) -> pd.DataFrame:
    work = frame.reindex(columns=ANCHOR_KEY_COLUMNS + [target_column]).copy()
    work['anchor_id'] = make_anchor_id(frame).to_numpy()
    work['group_h'] = np.maximum(np.asarray(h_values, dtype=float), 0.0)
    work['group_increment'] = np.maximum(
        pd.to_numeric(work[target_column], errors='coerce').to_numpy(dtype=float)
        - np.asarray(base_prediction_at_x0, dtype=float),
        0.0,
    )
    work = work.replace([np.inf, -np.inf], np.nan).dropna(subset=['anchor_id', 'group_h', 'group_increment'])
    rows = []
    for anchor_id, group in work.groupby('anchor_id', sort=True):
        group_h = group['group_h'].to_numpy(dtype=float)
        group_increment = group['group_increment'].to_numpy(dtype=float)
        denominator = float(np.sum(group_h * group_h))
        if not np.isfinite(denominator) or denominator <= ANCHOR_H_EPS ** 2:
            continue
        numerator = float(np.sum(group_h * group_increment))
        first = group.iloc[0]
        rows.append({
            'anchor_id': anchor_id,
            'date': first['date'], 'sym': first['sym'], 'side': first['side'],
            'quote_strategy': first['quote_strategy'],
            'A_label': max(numerator / denominator, 0.0),
            'n_curve_points': int(len(group)),
            'h_sum_squares': denominator,
            'label_fit': 'nonnegative_curve_least_squares',
            'anchor_granularity': ANCHOR_GRANULARITY,
        })
    return pd.DataFrame(rows)


base_prediction_at_x0 = {}
shape_models = {}
shape_grids = {}
shape_grid_diagnostics = []
anchor_strength_labels = {}
anchor_condition_frame = train[ANCHOR_KEY_COLUMNS + condition_features].copy()
anchor_condition_frame['anchor_id'] = make_anchor_id(anchor_condition_frame).to_numpy()
anchor_condition_frame = anchor_condition_frame.sort_values('anchor_id').drop_duplicates('anchor_id', keep='first').reset_index(drop=True)
for quantile in QUANTILE_REGISTRY:
    base_prediction_at_x0[quantile] = np.maximum(direct_models[quantile].predict(X_train_x0), 0.0)
    increment_bad_bps = np.maximum(y_total_train.to_numpy() - base_prediction_at_x0[quantile], 0.0)
    shape_total_train = increment_bad_bps
    h_grid = build_quantile_shape_grid(train, increment_bad_bps, quantile)
    h_values, shape_h_support_status = interpolate_quantile_shape(train, h_grid)
    anchor_labels = fit_anchor_strength_label(train, h_values, base_prediction_at_x0[quantile], 'total_bad_move_bps')
    if anchor_labels.empty:
        raise ValueError(f'no valid anchor-level A label for quantile={quantile}')
    anchor_strength_labels[quantile] = anchor_labels.copy()
    anchor_training = anchor_condition_frame.merge(anchor_labels[['anchor_id', 'A_label']], on='anchor_id', how='inner', validate='one_to_one')
    X_anchor_condition_train = make_matrix(anchor_training, condition_features)
    strength_target = anchor_training['A_label'].to_numpy(dtype=float)
    strength_model = LGBMRegressor(**model_parameters(quantile))
    strength_model.fit(X_anchor_condition_train, strength_target)
    shape_models[quantile] = strength_model
    shape_grids[quantile] = h_grid.reset_index(drop=True)
    shape_grid_diagnostics.append(h_grid.assign(shape_h_mode=SHAPE_H_MODE, side_aware=True))
pd.concat(shape_grid_diagnostics, ignore_index=True).to_csv(OUTPUT_DIR / '02_shape_curve_diagnostics.csv', index=False)
pd.concat([labels.assign(quantile=quantile, quantile_label=quantile_label(quantile)) for quantile, labels in anchor_strength_labels.items()], ignore_index=True).to_csv(OUTPUT_DIR / '02_anchor_strength_labels.csv', index=False)


In [ ]:
prediction_keys = ['date', 'sym', 'side', 'quote_strategy', 'x_adv', 'ratio_bucket', 'total_bad_move_bps', 'impact_me_bad_bps']
prediction_frame = pd.concat([
    calibration[prediction_keys].assign(split='calibration'),
    test[prediction_keys].assign(split='test'),
], ignore_index=True)
X_prediction = make_matrix(prediction_frame)
prediction_x0 = prediction_frame.copy()
prediction_x0['x_adv'] = 0.0
X_prediction_x0 = make_matrix(prediction_x0)
X_condition_prediction = make_matrix(prediction_frame, condition_features)
for quantile in QUANTILE_REGISTRY:
    qn = quantile_label(quantile)
    prediction_frame[f'direct_lgbm_{qn}_raw'] = np.maximum(direct_models[quantile].predict(X_prediction), 0.0)
    h_grid = shape_grids[quantile]
    h_values, shape_h_support_status = interpolate_quantile_shape(prediction_frame, h_grid)
    base_prediction_at_x0 = np.maximum(direct_models[quantile].predict(X_prediction_x0), 0.0)
    strength = np.maximum(shape_models[quantile].predict(X_condition_prediction), 0.0)
    prediction_frame[f'shape_strength_{qn}_raw'] = np.maximum(base_prediction_at_x0 + strength * h_values, 0.0)
    prediction_frame[f'shape_h_support_status_{qn}'] = shape_h_support_status
prediction_frame['impact_guardrail_q95_raw'] = np.maximum(impact_model.predict(X_prediction), 0.0)


def target_coverage(quantile: float) -> float:
    return 0.955 if np.isclose(quantile, 0.95) else float(quantile)


def residual_buffer(values: pd.Series, target: float) -> float:
    clean = pd.to_numeric(values, errors='coerce').dropna()
    return max(float(clean.quantile(target)), 0.0) if len(clean) else 0.0


calibration_rows = []
cal_mask = prediction_frame['split'].eq('calibration')
calibration_date_values = sorted(prediction_frame.loc[cal_mask, 'date'].dropna().unique())
fold_edges = np.array_split(calibration_date_values, min(3, len(calibration_date_values)))
for family in ['direct_lgbm', 'shape_strength']:
    for quantile in QUANTILE_REGISTRY:
        qn = quantile_label(quantile)
        raw_col = f'{family}_{qn}_raw'
        residual = prediction_frame.loc[cal_mask, 'total_bad_move_bps'] - prediction_frame.loc[cal_mask, raw_col]
        global_buffer = residual_buffer(residual, target_coverage(quantile))
        causal_history = []
        prior_dates = []
        for fold_dates in fold_edges:
            if prior_dates:
                prior_mask = cal_mask & prediction_frame['date'].isin(prior_dates)
                prior_residual = prediction_frame.loc[prior_mask, 'total_bad_move_bps'] - prediction_frame.loc[prior_mask, raw_col]
                causal_history.append(residual_buffer(prior_residual, target_coverage(quantile)))
            prior_dates.extend(list(fold_dates))
        causal_floor = float(np.quantile(causal_history, 0.80)) if causal_history else 0.0
        for (side, bucket), index in prediction_frame.loc[cal_mask].groupby(['side', 'ratio_bucket']).groups.items():
            bucket_residual = prediction_frame.loc[index, 'total_bad_move_bps'] - prediction_frame.loc[index, raw_col]
            bucket_buffer = residual_buffer(bucket_residual, target_coverage(quantile)) if len(index) >= 20 else global_buffer
            calibration_rows.append({
                'model_family': family, 'quantile_level': quantile, 'side': side, 'ratio_bucket': bucket,
                'n': len(index), 'target_coverage': target_coverage(quantile),
                'calibration_buffer_bps': bucket_buffer, 'causal_floor_buffer_bps': causal_floor,
                'final_buffer_bps': max(bucket_buffer, causal_floor),
                'rolling_floor_final_month_excluded': True,
            })
        buffer_map = {(row['side'], row['ratio_bucket']): row for row in calibration_rows if row['model_family'] == family and np.isclose(row['quantile_level'], quantile)}
        buffers = [buffer_map.get((side, bucket), {'calibration_buffer_bps': global_buffer, 'final_buffer_bps': max(global_buffer, causal_floor)}) for side, bucket in zip(prediction_frame['side'], prediction_frame['ratio_bucket'])]
        prediction_frame[f'{family}_{qn}_calibrated'] = prediction_frame[raw_col] + np.array([row['calibration_buffer_bps'] for row in buffers])
        prediction_frame[f'{family}_{qn}_final'] = prediction_frame[raw_col] + np.array([row['final_buffer_bps'] for row in buffers])

impact_residual = prediction_frame.loc[cal_mask, 'impact_me_bad_bps'] - prediction_frame.loc[cal_mask, 'impact_guardrail_q95_raw']
impact_buffer = residual_buffer(impact_residual, IMPACT_GUARDRAIL_QUANTILE)
prediction_frame['impact_guardrail_q95_final'] = prediction_frame['impact_guardrail_q95_raw'] + impact_buffer
calibration_table = pd.DataFrame(calibration_rows)
calibration_table.to_csv(OUTPUT_DIR / '02_calibration_table.csv', index=False)


In [ ]:
def pinball_loss(y_true: pd.Series, y_pred: pd.Series, quantile: float) -> float:
    error = pd.to_numeric(y_true, errors='coerce') - pd.to_numeric(y_pred, errors='coerce')
    return float(np.nanmean(np.maximum(quantile * error, (quantile - 1.0) * error)))


metric_rows = []
group_rows = []
for family in ['direct_lgbm', 'shape_strength']:
    for quantile in QUANTILE_REGISTRY:
        qn = quantile_label(quantile)
        for stage in ['raw', 'calibrated', 'final']:
            pred_col = f'{family}_{qn}_{stage}'
            for split_name, mask in {
                'calibration': prediction_frame['split'].eq('calibration'),
                'test': prediction_frame['split'].eq('test'),
                'final_month': prediction_frame['split'].eq('test') & prediction_frame['date'].dt.month.eq(12),
            }.items():
                subset = prediction_frame.loc[mask]
                if subset.empty:
                    continue
                coverage = float((subset['total_bad_move_bps'] <= subset[pred_col]).mean())
                metric_rows.append({
                    'model_family': family, 'quantile_level': quantile, 'prediction_stage': stage,
                    'split': split_name, 'n': len(subset), 'coverage': coverage,
                    'target_coverage': target_coverage(quantile),
                    'pinball_loss': pinball_loss(subset['total_bad_move_bps'], subset[pred_col], quantile),
                    'mean_prediction_bps': float(subset[pred_col].mean()),
                    'final_month_learning_excluded': True,
                })
            final_test = prediction_frame[prediction_frame['split'].eq('test')]
            if stage == 'final':
                for (side, bucket), subset in final_test.groupby(['side', 'ratio_bucket'], dropna=False):
                    group_rows.append({
                        'model_family': family, 'quantile_level': quantile, 'prediction_stage': stage,
                        'side': side, 'ratio_bucket': bucket, 'n': len(subset),
                        'coverage': float((subset['total_bad_move_bps'] <= subset[pred_col]).mean()),
                        'target_coverage': target_coverage(quantile),
                        'pinball_loss': pinball_loss(subset['total_bad_move_bps'], subset[pred_col], quantile),
                    })

crossing_rows = []
for family in ['direct_lgbm', 'shape_strength']:
    for stage in ['raw', 'calibrated', 'final']:
        for lower, upper in zip(QUANTILE_REGISTRY[:-1], QUANTILE_REGISTRY[1:]):
            lower_col = f'{family}_{quantile_label(lower)}_{stage}'
            upper_col = f'{family}_{quantile_label(upper)}_{stage}'
            gap = prediction_frame[lower_col] - prediction_frame[upper_col]
            positive = gap[gap > 0]
            crossing_rows.append({
                'model_family': family, 'prediction_stage': stage,
                'lower_quantile': lower, 'upper_quantile': upper,
                'crossing_rate': float((gap > 0).mean()),
                'mean_crossing_bps': float(positive.mean()) if len(positive) else 0.0,
                'p95_crossing_bps': float(positive.quantile(0.95)) if len(positive) else 0.0,
            })

metrics = pd.DataFrame(metric_rows)
grouped_coverage = pd.DataFrame(group_rows)
crossing_diagnostics = pd.DataFrame(crossing_rows)
metrics.to_csv(OUTPUT_DIR / '02_model_metrics.csv', index=False)
grouped_coverage.to_csv(OUTPUT_DIR / '02_grouped_coverage.csv', index=False)
crossing_diagnostics.to_csv(OUTPUT_DIR / '02_crossing_diagnostics.csv', index=False)
prediction_frame.to_parquet(OUTPUT_DIR / '02_predictions.parquet', index=False, compression='zstd')
bundle = {
    'quantile_registry': QUANTILE_REGISTRY,
    'impact_guardrail_quantile': IMPACT_GUARDRAIL_QUANTILE,
    'feature_columns': feature_columns, 'condition_features': condition_features, 'medians': medians,
    'direct_models': direct_models, 'impact_model': impact_model,
    'shape_strength_models': shape_models, 'shape_grids': shape_grids,
    'anchor_strength_labels': anchor_strength_labels,
    'anchor_key_columns': ANCHOR_KEY_COLUMNS,
    'strength_label_granularity': STRENGTH_LABEL_GRANULARITY,
    'base_label_source': BASE_LABEL_SOURCE,
    'shape_h_mode': SHAPE_H_MODE, 'shape_h_side_aware': True,
    'shape_reference_x_adv': SHAPE_REFERENCE_X_ADV,
    'shape_min_reference_scale_ratio': SHAPE_MIN_REFERENCE_SCALE_RATIO,
    'calibration_table': calibration_table, 'impact_buffer_bps': impact_buffer,
    'run_mode': RUN_MODE, 'random_seed': RANDOM_SEED,
    'no_order_anchor_source': 'model_prediction_at_x0',
}
joblib.dump(bundle, MODEL_DIR / '02_model_bundle.joblib')
stage_contract = {
    'run_mode': RUN_MODE, 'evaluation_scope': STAGE['evaluation_scope'],
    'quantile_registry': QUANTILE_REGISTRY,
    'cross_quantile_order_enforced': False, 'final_month_learning_excluded': True,
    'strength_label_granularity': STRENGTH_LABEL_GRANULARITY,
    'base_label_source': BASE_LABEL_SOURCE,
    'shape_h_mode': SHAPE_H_MODE, 'shape_h_side_aware': True,
    'shape_min_reference_scale_ratio': SHAPE_MIN_REFERENCE_SCALE_RATIO,
    'shape_curves': 'outputs/02_shape_curve_diagnostics.csv',
    'anchor_strength_labels': 'outputs/02_anchor_strength_labels.csv',
    'no_order_anchor_source': 'model_prediction_at_x0',
    'model_bundle': 'models/02_model_bundle.joblib', 'predictions': 'outputs/02_predictions.parquet',
}
(OUTPUT_DIR / '02_stage_contract.json').write_text(json.dumps(stage_contract, ensure_ascii=False, indent=2), encoding='utf-8')
print(metrics.tail(12).to_string(index=False))
print(crossing_diagnostics.to_string(index=False))
